In [16]:
import os
from pathlib import Path
from dataclasses import dataclass
from src.TALOS.constants import *
from src.TALOS.utils.common import read_yaml,create_directories
from src.TALOS import logger

In [40]:
ls

Dockerfile        config/           research/         templates/
LICENSE           logs/             schema.yaml       venv/
README.md         main.py           setup.py
app.py            params.yaml       src/
artifacts/        requirements.txt  template.py


In [4]:
cd ..

/Users/gojuruakshith/Tactical-Aerospace-Localization-Objective-Scoring-TALOS-


In [7]:
@dataclass(frozen=True)
class DataValidationConfig:
  root_dir: Path
  unzip_data_dir: Path
  STATUS_FILE: Path

In [50]:
class ConfigurationManager:
    def __init__(self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    def get_data_validation_config(self)->DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])
        return DataValidationConfig(
            root_dir= config.root_dir,
            unzip_data_dir= config.unzip_data_dir,
            STATUS_FILE= config.STATUS_FILE,
        )

In [51]:
class DataValidation:
    def __init__(self,config: DataValidationConfig):
        self.config = config
    def validate_all_files_exist(self):
        try:
            validation_status = True
            root_path = self.config.unzip_data_dir
            for folder in ["train", "test"]:
                folder_path = os.path.join(root_path, folder)
                if not os.path.exists(folder_path):
                    logger.error(f"Folder missing: {folder_path}")
                    print("Folder missing")
                    validation_status = False
                    continue
                all_files = os.listdir(folder_path)
                images = {os.path.splitext(f)[0] for f in all_files if f.endswith('.png')}
                xmls = {os.path.splitext(f)[0] for f in all_files if f.endswith('.xml')}
                missing_xml = images - xmls
                missing_png = xmls - images

                if missing_xml:
                    logger.error(f"In {folder}: Missing .xml for {missing_xml}")
                    validation_status = False

                if missing_png:
                    logger.error(f"In {folder}: Missing .png for {missing_png}")
                    validation_status = False
            os.makedirs(os.path.dirname(self.config.STATUS_FILE), exist_ok=True)
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation Status: {validation_status}")

            return validation_status

        except Exception as e:
            logger.error(f"Validation process failed: {e}")
            raise e

In [53]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config = data_validation_config)
    data_validation.validate_all_files_exist()
except Exception as e:
    logger.error(f"Validation process failed: {e}")

[2026-02-17 18:20:39,844: INFO: common: yaml_path: config/config.yaml loaded successfully]
[2026-02-17 18:20:39,845: INFO: common: yaml_path: params.yaml loaded successfully]
[2026-02-17 18:20:39,846: INFO: common: yaml_path: schema.yaml loaded successfully]
[2026-02-17 18:20:39,847: INFO: common: created directory at: artifacts]
[2026-02-17 18:20:39,848: INFO: common: created directory at: artifacts/data_validation]
True
